In [1]:
from dotenv import load_dotenv
import os 
import sys

load_dotenv(dotenv_path='/Users/munizasiddiqui/easyf1/.env')
API_KEY = os.getenv("GEMINI_API_KEY")
sys.path.append('/Users/munizasiddiqui/easyf1/')

print(f"API Key Loaded: {'YES' if API_KEY else 'NO'}")

API Key Loaded: YES


In [2]:
#Verifying the compatibility with the system 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

#creating the agent brain the LLM,
# it is the langchain wrapper around gemini
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    google_api_key = API_KEY,
    temperature = 0,
    timeout = 30
) 

#defining a throwaway testtool 
@tool
def add_numbers(a: int, b: int) -> int:
    """Adding numbeers and return the sum"""
    return a + b

#binding the tools to model, as it is the important step.
#without this the model doesnt know the tool exists
llm_with_tools = llm.bind_tools([add_numbers])

#getting the response
response = llm_with_tools.invoke("what is 334 + 545")

print (f"Tool calls: {response.tool_calls}")
print (f"Content: {response.content}")

/Users/munizasiddiqui/easyf1/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Tool calls: [{'name': 'add_numbers', 'args': {'a': 334, 'b': 545}, 'id': '9953e99b-01ca-4248-8809-d79770981306', 'type': 'tool_call'}]
Content: 


In [ ]:
#running the tool and feeding the result back
from langchain_core.messages import HumanMessage, ToolMessage
messages = [HumanMessage(content="what is 847 plus 293?")]
response = llm_with_tools.invoke(messages)
messages.append(response)

print("Step 1: Model's decision")
print(f" Wants to call: {response.tool_calls}")
print()

#step 2, actually running the tool model ask for
for tool_call in response.tool_calls:
    
    #getting the arguments the model chose
    args = tool_call['args']

    #running the actual python funvtion with those arguments 
    result = add_numbers.invoke(args)

    print(f"step 2, we ran the tool")
    print(f" add_numbers({args}) = {result}")
    print()

    #packaging the result as ToolMessage and add to conversation
    messages.append(ToolMessage(
        content = str(result),
        tool_call_id = tool_call['id']
        ))
    
    #Give the result back to model so it can answer
    final_response = llm_with_tools.invoke (messages)

    print("Step 3: Models final answer with result:")
    print(f"{final_response.content}")



Step 1: Model's decision
 Wants to call: [{'name': 'add_numbers', 'args': {'b': 293, 'a': 847}, 'id': 'de2754a8-1636-49a2-add6-361ce8d7796d', 'type': 'tool_call'}]

step 2, we ran the tool
 add_numbers({'b': 293, 'a': 847}) = 1140

Step 3: Models final answer with result:



In [5]:
print(f"Content: {final_response.content}")
print(f"Type: {type(final_response.content)}")
print(f"Full response: {final_response}")

Content: 
Type: <class 'str'>
Full response: content='' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019fbe61-e792-7ab1-8278-e4fc007913b7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 106, 'output_tokens': 0, 'total_tokens': 106, 'input_token_details': {'cache_read': 0}}


In [3]:
# Cell A — minimal setup
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path='/Users/munizasiddiqui/easyf1/.env')
API_KEY = os.getenv("GEMINI_API_KEY")
print("Key loaded:", bool(API_KEY))

Key loaded: True


In [4]:
# Cell B — raw client test
from google import genai
raw_client = genai.Client(api_key=API_KEY)
print("Calling Gemini...")
response = raw_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Say hello"
)
print(response.text)

Calling Gemini...
Hello!


In [5]:
#AGENT LLM SETUP
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=API_KEY,
    temperature=0
)

@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together and returns the sum."""
    return a + b

llm_with_tools = llm.bind_tools([add_numbers])

print("Agent LLM ready ✅")

Agent LLM ready ✅


In [6]:
#Testing does the agent request the tool
response = llm_with_tools.invoke("What is 847 plus 293?")

print(f"Tool calls: {response.tool_calls}")
print(f"Content: {response.content}")

Tool calls: [{'name': 'add_numbers', 'args': {'b': 293, 'a': 847}, 'id': 'b5b15499-1f0c-49eb-be6f-f2e2a847c9b4', 'type': 'tool_call'}]
Content: 


In [ ]:
#Load RAG essentials
#ChromaDB + embed_ text + query function, for agent tools to use

from google import genai 
import chromadb 
from fastembed import TextEmbedding

#Gemini client (raw, for embeddings independent use)
gclient = genai.Client(api_key = API_KEY)

#Loading embedding model
embedding_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    cache_dir="/Users/munizasiddiqui/easyf1/model_cache"
)

#connecting to existing ChromaDB
chroma_client = chromadb.PersistentClient(path = '/Users/munizasiddiqui/easyf1/chromadb')
collection = chroma_client.get_collection(name = "easyf1_races")

def embed_text(text):
    """Embeding text locally using fastembed"""
    return list (embedding_model.embed([text])) [0].tolist()

print(f"RAG essentials loaded")
print(f"chunks in ChromaDB: {collection.count()}")



2026-08-13 18:03:36.751 | WARNING  | fastembed.common.model_management:download_files_from_huggingface:223 - Local file sizes do not match the metadata.


NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from /var/folders/0n/hylvz_k92g321017w4jc9st40000gn/T/fastembed_cache/models--qdrant--bge-small-en-v1.5-onnx-q/snapshots/52398278842ec682c6f32300af41344b1c0b0bb2/model_optimized.onnx failed:Load model /var/folders/0n/hylvz_k92g321017w4jc9st40000gn/T/fastembed_cache/models--qdrant--bge-small-en-v1.5-onnx-q/snapshots/52398278842ec682c6f32300af41344b1c0b0bb2/model_optimized.onnx failed. File doesn't exist

In [24]:
#Building the three agent tools 
#these tools will be choosen by the agent for calling 
#docstring is what the agent reads to decide 

from langchain_core.tools import tool 

#Tool 1: RAG Retrieval
@tool
def search_race_data(query: str) -> str:
    """
    it will search the historical F1 race data to find the corresponding drivers info,
    tyres, pits stops, safety car and lap details.
    will be using this for real race facts like tyre age, what happened on a specific lap,
    ehat compund a driver was on
    
    Args: 
        query: A description of the race data we need, eg: Leclerc tyre age at lap 20 Baku
    """

    question_embedding = embed_text(query)
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results= 5,
        include = ['documents']
    )

    chunks = results['documents'][0]
    return "\n\n".join(chunks)

#Tool 2: The Pit Window Calculator
@tool
def calculate_pit_loss(safety_car_active: bool, circuit:str = "Silverstone") -> str:
    """
    This function will calculate time consumed a pit stop, in seconds
    
    Args: 
        safety_car_active: True if a safety car or VSC is currently deployed
        circuit: The name of the circuit
    """
#Approximate pit lane time loss by circuit in seconds
    circuit_pit_loss = {
        "Silverstone": 22.0,
        "Monza": 19.0,
        "Monaco": 24.0,
        "Spa": 21.0
    }

    base_loss = circuit_pit_loss.get(circuit, 22.0)

    if safety_car_active:
        #As the field is slower under the SC, so the relative pitstop reduces t0 70%
        actual_loss = base_loss * 0.3
        condition = "under safety car"

    else:
        actual_loss = base_loss
        condition = "under green flag"

    return (f"Pit loss at {circuit} {condition}: "
            f"{actual_loss:.1f} seconds"
            f"(green flag baseline: {base_loss:.1f}s).")

#Tool 3: Tyre Degradation Estimator
@tool
def assess_tyre_condition(compound: str, tyre_age: int) -> str:
    """
    assess tyre conditon, whether it's within optimal life (cliff) or past it's optimal window.
    will using this to decide drivers needs new tyre compound/replacing
    
    Args:
        compound: Soft, Medium, Hard
        tyre_age: no. of laps tyre has done
    """
    #Approximate cliff points for each compound
    cliff = {
        "SOFT" : 18,
        "MEDIUM": 28,
        "HARD": 40
    }

    compound_upper = compound.upper()
    cliff_point = cliff.get(compound_upper, 25)

    if tyre_age >= cliff_point:
        status = "CRITICAL - past the cliff, major performance loss"

    elif tyre_age >= cliff_point - 5:
        status = "WARNING - approaching the cliff, degrading fast"

    else:
        status = "GOOD - still within optimal window"

    return(f"{compound_upper} tyres at {tyre_age} laps: {status}."
           f"Typical cliff for {compound_upper} is around {cliff_point} laps.")

print("Three EasyF1 tools ready")
print("1- search_race_data -- RAG Retrieval")
print("2- calculate_pit_loss -- Pit loss maths")
print("3- assess_tyre_condition -- RAG Retrieval")


Three EasyF1 tools ready
1- search_race_data -- RAG Retrieval
2- calculate_pit_loss -- Pit loss maths
3- assess_tyre_condition -- RAG Retrieval


In [25]:
#Building the LangGraph agent
#this will wire the tool + LLM into a ReAct loop 
#Working --> agent thinks, tool calls, observe results, thinks again

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing import Annotated, TypedDict
from langchain_core.messages import SystemMessage

#step 1 agent tools list 
tools = [search_race_data, calculate_pit_loss, assess_tyre_condition]

#binding tools LLM, this will tell LLM which tools s has access to 
agent_llm = llm.bind_tools(tools)

#Step3 Definin the agent state
#state is the memory that flows through the whole loop 
#add message will append new message rather the overwrite 

class AgentState (TypedDict):
    messages: Annotated[list, add_messages]

#Step4, defining the agent node, this will be the thinking step which calls the LLM 
def call_agent(state:AgentState):
    #system prompts will guide the agents behavior
    system = SystemMessage(content = """You are EasyF1, a F1 race strategy assistant

when given a race scenario, reason step by step:
1. Use search_race_data to get real facts about the driver's tyres and situation
2. Use assess_tyre_condition to check if the tyres are past their cliff
3. Use calculate_pit_loss to determine the time cost of pitting
4. Then give your final recommendation

Always base your recommendation on the tool results, not general knowledge.

End with a clear recommendation in this format:
VERDICT: [PIT NOW / STAY OUT / PIT IN N LAPS]
CONFIDENCE: [LOW / MEDIUM / HIGH]
REASONING: [2-3 sentences citing the specific tool results]""")

    #calling the LLM with the system prompt + conversation so far
    messages = [system] + state["messages"]
    response = agent_llm.invoke(messages)

    #returning response to add to state
    return{"messages": [response]}

#step5 defining the conditional edge
#after the agent thinks, decide: call tools or finish
def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    #if the agent requested tools, it will run them 
    if last_message.tool_calls:
        return "tools"
    #or the agent is done
    return END

#Step6: Assemble the graph
graph = StateGraph(AgentState)

# Add the two nodes
graph.add_node("agent", call_agent)
graph.add_node("tools", ToolNode(tools))

# Set where the graph starts
graph.set_entry_point("agent")

# Add the conditional edge from agent
graph.add_conditional_edges("agent", should_continue)

# THE LOOP: after tools run, go back to the agent
graph.add_edge("tools", "agent")

# Compile it into a runnable app
easyf1_agent = graph.compile()

print("EasyF1 agent compiled ✅")

EasyF1 agent compiled ✅


In [26]:
#Running the agent + watching it reason 
#streaming the steps so we can see the agent's thinking, which tools it calls, what is observes,
#and how it concludes

from langchain_core.messages import HumanMessage

question = "It's lap 3 at silverstone. Hmailton is on 33 lap old medium tyres. A safety car has just been deployed. What should mercedes do?"

print(f"Question: {question}")
print("=" * 50)
print()

#streaming each step of agent's reasoning
for step in easyf1_agent.stream ({"messages": [HumanMessage(content = question)]}):
    for node_name, node_output in step.items():

        last_msg = node_output["messages"][-1]

        #if it the agent node 
        if node_name =="agent":
            #did he agent request tools?
            if last_msg.tool_calls:
                print(f"Agent thinking wants to call tools")
                for tc in last_msg.tool_calls:
                    print(f" Action: {tc['name']} ({tc['args']})")

            else:
                #final answer
                print(f"Agent's final answer")
                print(f"{last_msg.content}")

        #if it's the tools node 
        elif node_name == "tools":
            print(f"Tool Results:")
            print(f" {last_msg.content[:300]}...")
            print()


Question: It's lap 3 at silverstone. Hmailton is on 33 lap old medium tyres. A safety car has just been deployed. What should mercedes do?

Agent thinking wants to call tools
 Action: assess_tyre_condition ({'compound': 'Medium', 'tyre_age': 33})
 Action: calculate_pit_loss ({'circuit': 'Silverstone', 'safety_car_active': True})
Tool Results:
 Pit loss at Silverstone under safety car: 6.6 seconds(green flag baseline: 22.0s)....

Agent's final answer
The tyres are critically worn and past their optimal window. Pitting now under the safety car would minimize the time loss, costing only 6.6 seconds compared to a normal pit stop.

VERDICT: PIT NOW
CONFIDENCE: HIGH
REASONING: Hamilton's medium tyres are critically worn at 33 laps, past their performance cliff. Pitting under the safety car will only cost 6.6 seconds, making it an opportune moment to change tyres.
